# Проверка SFT: исходная T-lite против T-lite с адаптером (EXP-015)

Решение 03.11.2026 (`docs/decisions.md`, раздел 9): главная проверка SFT — генерация на 25 темах validation в обоих
режимах против исходной T-lite. По одному ответу на тему и режим (T = 0.9, seed по теме — одинаковый у обеих моделей),
плюс по 4 ответа без поиска на 6 темах — разброс награды внутри темы (на нём учится GRPO). Отдельно — loss обеих
моделей на 16 отложенных примерах SFT (точка отсчёта для loss обучения).

Каждая модель — своей ячейкой (`run_model`), сравнение — последней (`compare`); вывод каждой ячейки остаётся.
Карту выбирает аргумент `device`. Эпизоды можно поделить между двумя картами: в копии тетрадки та же модель
с `part=(1, 2)` на другой карте, здесь — `part=(0, 2)`. Готовые эпизоды при перезапуске пропускаются. Результат —
`runs/sft_eval/<RUN>/<модель>/` (эпизоды, ответы, pptx), `runs/sft_eval/<RUN>/comparison.csv` и `episodes.csv`.

In [ ]:
import csv, dataclasses, gc, json, random, re, statistics, sys, time, traceback, os
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
import torch
from presentation_rl.dsl import parse
from presentation_rl.environment import build_prompt, run_episode
from presentation_rl.model_loader import hf_generator, load_model
from presentation_rl.perturb import task_context
from presentation_rl.prompts import system_prompt
from presentation_rl.rendering import build_presentation, pick_theme
from presentation_rl.rewards import Lemmatizer, SentenceEmbedder, load_key_points, score
from presentation_rl.tasks import load_tasks
from presentation_rl.tools.cache import WikiCache
from presentation_rl.tools.wiki import WikiClient
from presentation_rl.tools.wiki_tools import WikiTools
from presentation_rl.training import sft

RUN = "20261107"  # общая папка прогона обеих моделей
config = sft.SFTConfig.load(ROOT / "configs" / "experiments" / "sft_tlite.yaml")  # модель и отложенные темы
SEED = 0
DIVERSITY_TOPICS, DIVERSITY = 6, 4  # темы с тезисами и ответов на тему без поиска — для разброса награды
CONTACT = os.environ.get("WIKI_CONTACT", "https://github.com/doshik722")
cache = WikiCache(ROOT / "data" / "sources" / "wiki_cache", WikiClient(contact=CONTACT))
tasks = {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json", split="validation")}
points = load_key_points(ROOT / "data" / "tasks" / "key_points" / "key_points.json")
IDS = sorted(tasks)
WITH_POINTS = [key for key in IDS if key in points]
EPISODES = [(mode, key, 0) for mode in ("search", "plain") for key in IDS]
EPISODES += [("plain", key, s) for key in WITH_POINTS[:DIVERSITY_TOPICS] for s in range(1, DIVERSITY)]
print(f"тем {len(IDS)}, с тезисами {len(WITH_POINTS)}; эпизодов на модель {len(EPISODES)}; карт {torch.cuda.device_count()}")

## Функции генерации

`run_model` загружает модель на карту `device` (с адаптером, если он задан), проверяет поиск, считает loss
на отложенных примерах SFT (только при `part[0] == 0`), генерирует свою часть эпизодов и освобождает память.
Эпизод сохраняется сразу: `<режим>_<id>_<номер>.txt` — весь эпизод, `_answer.txt` — ответ, `.json` — время, конец,
вызовы, ошибка; ошибка эпизода прогон не останавливает.

In [ ]:
def heldout_loss(model, tokenizer, prompt):
    """Средний loss по репликам модели на отложенных примерах SFT — как eval_loss при обучении."""
    samples, _ = sft.drop_long(sft.build_samples(ROOT, tokenizer, prompt), config.max_tokens)
    _, held = sft.split(samples, config.eval_tasks, config.seed)
    losses = []
    with torch.no_grad():
        for sample in held:
            inputs = {k: v.to(model.device) for k, v in sft.collate([sample]).items()}
            losses.append(float(sft.masked_loss(model, inputs)))
    return {"examples": len(held), "topics": sorted({s.id for s in held}), "loss": statistics.mean(losses)}


def run_episode_to(out, mode, key, sample, tokenizer, generate, prompt):
    stem = f"{mode}_{key}_{sample}"
    if (out / f"{stem}.json").exists():
        return None
    request = dataclasses.replace(tasks[key].request, search=(mode == "search"))
    start = time.time()
    row = {"mode": mode, "id": key, "sample": sample}
    try:
        tools = WikiTools(cache) if request.search else None
        prompt_ids = prompt(request)
        torch.manual_seed(SEED + 1000 * sample + IDS.index(key))
        episode = run_episode(prompt_ids, tokenizer, generate, tools, max_new_tokens=8000)
        (out / f"{stem}.txt").write_text(tokenizer.decode(episode.ids[len(prompt_ids):]), encoding="utf-8")
        (out / f"{stem}_answer.txt").write_text(episode.answer, encoding="utf-8")
        row.update(finish=episode.finish, generated_tokens=episode.generated_tokens, calls=len(episode.calls), error="")
    except Exception:
        row.update(error=traceback.format_exc()[-1500:])
    row["seconds"] = round(time.time() - start)
    (out / f"{stem}.json").write_text(json.dumps(row, ensure_ascii=False, indent=1), encoding="utf-8")
    return row


def run_model(name, device, adapter=None, part=(0, 1)):
    """Ответы модели ``name`` («base» или «sft») на карте ``device``; ``adapter`` — путь к адаптеру от корня проекта."""
    out = ROOT / "runs" / "sft_eval" / RUN / name
    out.mkdir(parents=True, exist_ok=True)
    torch.cuda.set_device(device)
    start = time.time()
    model, tokenizer = load_model(config.model_id, config.revision, dtype="float16", device_map={"": device})
    if adapter:
        from peft import PeftModel

        model = PeftModel.from_pretrained(model, ROOT / adapter)
        model.eval()
    print(f"{name}: загрузка {time.time() - start:.0f} с, память {torch.cuda.memory_allocated(device) / 2**30:.1f} ГБ"
          + (f", адаптер {adapter}" if adapter else ""))
    # поиск работает? (ответ из кеша или из Википедии; ошибка здесь — эпизоды с поиском не пройдут)
    print(WikiTools(cache).call("search", "рентгеноструктурный анализ ДНК")[:400])

    def prompt(request):
        return build_prompt(tokenizer, system_prompt(request), request)

    if part[0] == 0:
        loss = {"model": name, "adapter": adapter, **heldout_loss(model, tokenizer, prompt)}
        (out / "heldout_loss.json").write_text(json.dumps(loss, ensure_ascii=False, indent=1), encoding="utf-8")
        print(f"{name}: loss на {loss['examples']} отложенных примерах SFT {loss['loss']:.4f}")
        torch.cuda.empty_cache()
    generate = hf_generator(model, tokenizer, temperature=0.9)
    mine = EPISODES[part[0]::part[1]]
    for k, (mode, key, sample) in enumerate(mine):
        row = run_episode_to(out, mode, key, sample, tokenizer, generate, prompt)
        if row:
            print(f"{k + 1}/{len(mine)} {mode}_{key}_{sample}: {row.get('finish', '—')}, токенов {row.get('generated_tokens', '—')}, "
                  f"вызовов {row.get('calls', '—')}, {row['seconds']} с" + (" — ОШИБКА" if row["error"] else ""), flush=True)
    del model, generate
    gc.collect()
    torch.cuda.empty_cache()
    print(f"{name}: готово за {(time.time() - start) / 60:.0f} мин")

## Исходная T-lite

In [ ]:
run_model("base", device=1)

## T-lite с адаптером SFT

Путь к адаптеру — выбранного запуска SFT (`runs/sft/<дата_время>/adapter`).

In [ ]:
run_model("sft", device=1, adapter="runs/sft/<дата_время>/adapter")

## Сравнение

Когда готовы ответы обеих моделей. По каждому ответу: доля корректных строк, число слайдов в ожидаемом диапазоне,
заключение последним слайдом, конец ответа, вызовы поиска (с поиском — были ли; без поиска — лишние), `#SOURCE`
из выдачи эпизода, награда и её компоненты. Разница SFT − исходная — по парам «тема, режим» с 95% интервалом
бутстрепа по темам; разброс награды внутри темы — по дополнительным ответам без поиска.

In [ ]:
CONCLUSION = re.compile(r"^\s*(вывод|выводы|заключение|итог|итоги)", re.IGNORECASE)


def evaluate(name, embedder, lemmatizer):
    folder = ROOT / "runs" / "sft_eval" / RUN / name
    rows = []
    for meta in sorted(folder.glob("*_*_*.json")):
        row = {"model": name, **json.loads(meta.read_text(encoding="utf-8"))}
        rows.append(row)
        if row["error"]:
            continue
        task, stem = tasks[row["id"]], meta.stem
        context = task_context(task, (folder / f"{stem}.txt").read_text(encoding="utf-8"), row["mode"] == "search")
        deck = parse((folder / f"{stem}_answer.txt").read_text(encoding="utf-8"))
        report = build_presentation(deck, folder / f"{stem}.pptx", pick_theme(task.request.theme, task.request.text),
                                    sources=(context["sources"] or {}).get, show_sources=task.request.sources)
        n = len(deck.slides)
        statuses = [i.status for i in report.items if i.kind == "SOURCE"]
        row.update(parse_rate=deck.parse_rate, slides=n, stop=row["finish"] == "stop",
                   conclusion=bool(n >= 3 and CONCLUSION.match(deck.slides[-1].title or "")),
                   searched=row["mode"] == "search" and row["calls"] > 0,
                   calls_in_plain=row["mode"] == "plain" and row["calls"] > 0,
                   sources_ok=statuses.count("ok"), sources_bad=statuses.count("bad_source"))
        if row["id"] in points:
            slide_range = tuple(context["slide_range"]) if context["slide_range"] else None
            r = score(deck, points[row["id"]], conclusion=context["conclusion"], slide_range=slide_range,
                      embedder=embedder, lemmatizer=lemmatizer, report=report, images_allowed=context["images_allowed"],
                      tool_text=context["tool_text"], calls=context["calls"], search=context["search"])
            low, high = r.expected_slides
            row.update(in_range=low <= n <= high, **{f"reward_{k}": v for k, v in r.components().items()})
    return rows


def paired(rows, metric, mode, n_boot=2000, seed=0):
    """Среднее у исходной и SFT и разница по парам (тема, режим) с 95% интервалом бутстрепа по темам."""
    get = {(r["model"], r["id"]): r.get(metric) for r in rows if r["mode"] == mode and r["sample"] == 0 and not r["error"]}
    keys = [k for k in IDS if get.get(("base", k)) is not None and get.get(("sft", k)) is not None]
    if not keys:
        return None
    diffs = [float(get[("sft", k)]) - float(get[("base", k)]) for k in keys]
    rng = random.Random(seed)
    boot = sorted(statistics.mean(rng.choice(diffs) for _ in diffs) for _ in range(n_boot))
    return {"mode": mode, "metric": metric, "n": len(keys),
            "base": statistics.mean(float(get[("base", k)]) for k in keys),
            "sft": statistics.mean(float(get[("sft", k)]) for k in keys), "diff": statistics.mean(diffs),
            "low": boot[int(0.025 * n_boot)], "high": boot[int(0.975 * n_boot)]}


def write_csv(path, items):
    fields = list(dict.fromkeys(k for r in items for k in r))
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fields)
        writer.writeheader()
        writer.writerows(items)


METRICS = ["parse_rate", "slides", "in_range", "conclusion", "stop", "searched", "calls_in_plain", "calls",
           "sources_ok", "sources_bad", "generated_tokens", "seconds", "reward_total", "reward_coverage",
           "reward_keys", "reward_length", "reward_visual", "reward_fake_quotes", "reward_format"]


def compare(device=1):
    embedder, lemmatizer = SentenceEmbedder("BAAI/bge-m3", device=f"cuda:{device}"), Lemmatizer()
    rows = evaluate("base", embedder, lemmatizer) + evaluate("sft", embedder, lemmatizer)
    folder = ROOT / "runs" / "sft_eval" / RUN
    write_csv(folder / "episodes.csv", rows)
    for name in ("base", "sft"):
        part = [r for r in rows if r["model"] == name]
        print(f"{name}: эпизодов {len(part)}, ошибок {sum(bool(r['error']) for r in part)}")
        path = folder / name / "heldout_loss.json"
        if path.exists():
            print(f"   loss на отложенных примерах SFT: {json.loads(path.read_text(encoding='utf-8'))['loss']:.4f}")
    table = [t for mode in ("search", "plain") for metric in METRICS if (t := paired(rows, metric, mode))]
    write_csv(folder / "comparison.csv", table)
    for t in table:
        print(f"{t['mode']:6s} {t['metric']:18s} n={t['n']:2d}  исходная {t['base']:8.3f}  SFT {t['sft']:8.3f}  "
              f"разница {t['diff']:+8.3f}  [{t['low']:+.3f}; {t['high']:+.3f}]")
    for name in ("base", "sft"):  # разброс награды внутри темы (без поиска): на этой разнице учится GRPO
        spreads = []
        for key in WITH_POINTS[:DIVERSITY_TOPICS]:
            values = [r["reward_total"] for r in rows if r["model"] == name and r["mode"] == "plain" and r["id"] == key
                      and not r["error"] and "reward_total" in r]
            if len(values) >= 2:
                spreads.append(statistics.pstdev(values))
        if spreads:
            print(f"разброс награды внутри темы, {name}: {statistics.mean(spreads):.4f} (тем {len(spreads)})")

In [ ]:
compare(device=1)